# Unit 3 acceptance — global DAC 10, local channel 2

The ordered local configuration maps Unit 1 to DAC 1–4, Unit 2 to DAC 5–8, and Unit 3 to DAC 9–12. The Keysight is physically wired to **DAC 10**. Only Unit 3 has the external analog supply. Units 1 and 2 have USB power only; their analog readback errors are expected, and these tests do not command their outputs.

Every test action and assertion is visible below and uses the same HTTP actions/state as the dashboard. Unit 3's outputs are isolated. Its firmware uses its own private MAC/IP and nominal calibration on all four channels. The ±0.5 mV accuracy limit is unchanged. Accuracy assertions are separate from functional assertions; failures are retained. No calibration is applied.

The preflight checks a small 0 → +0.05 → 0 V change before enabling the −9 to +10 V sweep. Safe zero also exercises small voltages on DAC 9, 11 and 12, with the current **0.02 V/s** rate. Every output/recovery action targets Unit 3. The last cell returns its outputs to their stored zero codes; the meter determines DAC 10's physical endpoint. USB emulation exercises the firmware button paths, not physical switch contacts.


In [ ]:
RUN_LIVE=False
ALL_OUTPUTS_ISOLATED=False
CHANNEL=10
BASE='http://127.0.0.1:8765'
TOLERANCE_V=.0005


In [ ]:
PREFLIGHT_PASSED=False
import json,time,uuid,re,math,statistics,csv
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import Request,urlopen
import matplotlib.pyplot as plt
assert RUN_LIVE and ALL_OUTPUTS_ISOLATED,'Enable both live-test flags first'
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'malab/dashboard').is_dir())
results=root/'malab/notebooks/results'/('modules_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
results.mkdir(parents=True)
token=re.search(r'name="dac-token" content="([^"]+)"',urlopen(BASE).read().decode()).group(1)
def state():return json.load(urlopen(BASE+'/api/state',timeout=3))
s=state();assert s['meter']['channel']==CHANNEL,'Meter label and physical connection must match'
UNIT=(CHANNEL-1)//4+1
assert UNIT==3 and CHANNEL==10
assert len(s['units'])==3 and len(s['channels'])==12
from malab.bench_config import load_bench_config
assert s["units"][UNIT-1]["address"]==load_bench_config()["units"][UNIT-1]["address"]
CHANNELS=list(range(4*(UNIT-1)+1,4*UNIT+1))
def row(s,ch=CHANNEL):return s['channels'][ch-1]
def unit(s):return s['units'][UNIT-1]
def codes(s):return [row(s,ch)['code'] for ch in CHANNELS]
def compact(s):return {k:s[k] for k in ('now','units','channels','meter')}
def wait_for(predicate,timeout=25):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        s=state()
        if predicate(s):return s
        time.sleep(.05)
    raise AssertionError('State deadline expired: '+json.dumps(unit(s)))
def action(name,**fields):
    payload=dict(action=name,unit=UNIT,request_id=str(uuid.uuid4()),**fields)
    request=Request(BASE+'/api/action',json.dumps(payload).encode(),headers={'Content-Type':'application/json','X-DAC-Token':token})
    job=json.load(urlopen(request,timeout=7))
    s=wait_for(lambda s:s['jobs'].get(job['id'],{}).get('state') in ('complete','failed'))
    job=s['jobs'][job['id']]
    assert job['state']=='complete',job
    return job
def fresh(s):
    assert unit(s)['network']['state']=='good',unit(s)
    assert s['now']-unit(s)['network']['last_ok']<2.5
    assert s['meter']['state']=='good' and s['now']-s['meter']['at']<2.5
    assert not unit(s)['device_error'],unit(s)['device_error']
def readings(count=5):
    samples=[];previous=None
    while len(samples)<count:
        s=state();fresh(s)
        if s['meter']['at']!=previous:samples.append(compact(s));previous=s['meter']['at']
        time.sleep(.04)
    return samples
def save(name,value):(results/(name+'.json')).write_text(json.dumps(value,indent=2),encoding='utf-8')
wait_for(lambda s:unit(s)['network']['state']=='good' and s['meter']['state']=='good')
action('initialize');time.sleep(.4)
initial=state();fresh(initial);save('initial',compact(initial))
profile=row(initial)['calibration'];STEP=profile['volts_per_code'];OFFSET=profile['offset_nanovolts']*1e-9
ZERO_CODES=[row(initial,ch)['calibration']['zero_code'] for ch in CHANNELS]
def code_for(v):return min(262143,max(0,int(math.floor((v-OFFSET)/STEP+.5))))
print('Results:',results,'channel:',CHANNEL,'unit:',UNIT,'profile:',profile)

# Validate the physical meter association before the wider sweep.
probe=[]
for target in [0,.05,0]:
    action('set',channel=CHANNEL,voltage=target);time.sleep(.4)
    samples=readings()
    probe.append(dict(target=target,code=row(samples[-1])['code'],displayed=row(samples[-1])['voltage'],
                      meter=statistics.mean(s['meter']['voltage'] for s in samples)))
    assert probe[-1]['code']==code_for(target),probe[-1]
save('preflight',probe)
assert abs((probe[1]['meter']-probe[0]['meter'])-.05)<.005,probe
assert abs(probe[2]['meter']-probe[0]['meter'])<.0005,probe
PREFLIGHT_PASSED=True
print('PASS: meter follows DAC 10 small step and returns; readings:',probe)


## V10 — Set/get versus the Keysight, unchanged calibration

In [ ]:
assert PREFLIGHT_PASSED,'DAC 10 physical preflight has not passed'
voltage_rows=[];baseline_other={ch:row(state(),ch)['code'] for ch in CHANNELS if ch!=CHANNEL}
for target in [-9,-5,-1,-.05,0,.05,1,5,9,10,0]:
    action('set',channel=CHANNEL,voltage=target);time.sleep(.4)
    samples=readings();measured=statistics.mean(s['meter']['voltage'] for s in samples)
    displayed=row(samples[-1])['voltage']
    result=dict(target=target,displayed=displayed,meter=measured,
        request_error_mV=(measured-target)*1000,display_error_mV=(measured-displayed)*1000)
    voltage_rows.append(result);save('voltage',voltage_rows);print(result)
    assert all(row(samples[-1],ch)['code']==code for ch,code in baseline_other.items()),'Another output changed'
assert max(abs(r['request_error_mV']) for r in voltage_rows)<=TOLERANCE_V*1000,'Requested voltage exceeds ±0.5 mV accuracy'
assert max(abs(r['display_error_mV']) for r in voltage_rows)<=TOLERANCE_V*1000,'Displayed voltage exceeds ±0.5 mV accuracy'


## R10 — Ramp timing and relative path (calibration errors remain visible)

In [ ]:
assert PREFLIGHT_PASSED,'DAC 10 physical preflight has not passed'
ramp_rows=[];traces=[]
for speed,span in [(.001,.005),(.01,.05),(.1,.5)]:
    for start,target in [(0,span),(span,0)]:
        action('set',channel=CHANNEL,voltage=start);time.sleep(.4)
        before=state();fresh(before);start_v=row(before)['voltage'];start_meter=before['meter']['voltage']
        expected=math.ceil(abs(code_for(target)-row(before)['code'])/10)*math.ceil(10*STEP/speed*1e6)/1e6
        started=time.time();action('ramp',channel=CHANNEL,voltage=target,speed=speed,step_size=10)
        trace=[];deadline=time.monotonic()+expected+10
        while time.monotonic()<deadline:
            s=state();fresh(s);trace.append(compact(s))
            if unit(s)['ramp']['state']=='COMPLETE':break
            assert unit(s)['ramp']['state']=='RUNNING',unit(s)['ramp']
            time.sleep(.05)
        duration=time.time()-started
        assert unit(trace[-1])['ramp']['state']=='COMPLETE'
        active=[s for s in trace if unit(s)['ramp']['state']=='RUNNING']
        assert len({row(s)['code'] for s in active})>=3
        direction=1 if target>start else -1;last_v=start_v;last_m=None
        for s in active:
            v=row(s)['voltage'];m=s['meter']['voltage']
            assert direction*(v-last_v)>=-1e-9
            if last_m is not None:assert direction*(m-last_m)>=-TOLERANCE_V
            assert abs((m-start_meter)-(v-start_v))<=10*STEP+speed*.6+TOLERANCE_V
            last_v=v;last_m=m
        assert abs(duration-expected)<=expected*.05+.6,(duration,expected)
        time.sleep(.3);end=readings(3);measured=statistics.mean(s['meter']['voltage'] for s in end)
        result=dict(speed=speed,start=start,target=target,expected_s=expected,observed_s=duration,
                    meter_endpoint=measured,endpoint_error_mV=(measured-target)*1000)
        ramp_rows.append(result);traces.append(dict(summary=result,samples=trace));save('ramps',traces);print(result)
fig,axes=plt.subplots(3,1,figsize=(10,9))
for ax,entry in zip(axes,traces[::2]):
    t=entry['samples'];t0=t[0]['now']
    ax.plot([row(s)['at']-t0 for s in t],[row(s)['voltage'] for s in t],label='Device register volts')
    ax.plot([s['meter']['at']-t0 for s in t],[s['meter']['voltage'] for s in t],label='Keysight')
    ax.set(xlabel='Seconds',ylabel='V',title=str(entry['summary']['speed'])+' V/s');ax.legend()
fig.tight_layout();fig.savefig(results/'ramps.png',dpi=150);plt.show()
print('PASS: ramp timing, monotonic relative paths and changing telemetry')


In [ ]:
assert max(abs(r['endpoint_error_mV']) for r in ramp_rows)<=TOLERANCE_V*1000,'Ramp endpoints exceed ±0.5 mV'
print('PASS: absolute ramp endpoint accuracy')


## N10 — Unit network reset, disconnect and stop hold

In [ ]:
assert PREFLIGHT_PASSED,'DAC 10 physical preflight has not passed'
action('set',channel=CHANNEL,voltage=.05);time.sleep(.4);before=state();fresh(before)
reset=action('network')
after=wait_for(lambda s:unit(s)['network']['state']=='good' and unit(s)['network']['last_ok']>reset['finished'])
assert codes(after)==codes(before)
assert abs(after['meter']['voltage']-before['meter']['voltage'])<=TOLERANCE_V
action('disconnect');time.sleep(2.7);disconnected=state()
assert unit(disconnected)['network']['state']=='disconnected'
assert disconnected['now']-unit(disconnected)['network']['last_ok']>2.5
for other_unit in disconnected['units'][:UNIT-1]:
    assert other_unit['network']['state']=='good',other_unit
    assert disconnected['now']-other_unit['network']['last_ok']<2.5,other_unit
assert abs(disconnected['meter']['voltage']-before['meter']['voltage'])<=TOLERANCE_V
action('connect');wait_for(lambda s:unit(s)['network']['state']=='good')
action('ramp',channel=CHANNEL,voltage=0,speed=.01);time.sleep(1.3)
assert unit(state())['ramp']['state']=='RUNNING'
action('abort');held=wait_for(lambda s:unit(s)['network']['state']=='good' and unit(s)['ramp']['state']=='ABORTED')
time.sleep(.4);held=state();time.sleep(.8);later=state()
assert 0<row(held)['voltage']<.05
assert codes(later)==codes(held)
assert abs(later['meter']['voltage']-held['meter']['voltage'])<=TOLERANCE_V
save('network',dict(before=compact(before),after=compact(after),disconnected=compact(disconnected),held=compact(later)))
print('PASS: reset holds all four codes and measured output; disconnect ages; abort holds')


## S10 — Safe zero while TCP is disconnected; absolute zero checked separately

In [ ]:
assert PREFLIGHT_PASSED,'DAC 10 physical preflight has not passed'
safe_rows=[]
for sign in [1,-1]:
    for ch,v in zip([ch for ch in CHANNELS if ch!=CHANNEL],[.01,-.01,.02]):action('set',channel=ch,voltage=v)
    action('set',channel=CHANNEL,voltage=sign*.05);time.sleep(.4)
    before=state();action('disconnect')
    started=time.time();action('safe_zero');trace=[];deadline=time.monotonic()+10
    while time.monotonic()<deadline:
        s=state();trace.append(compact(s))
        assert unit(s)['network']['state']!='good'
        assert unit(s)['recovery']['state'] in ('ACTIVE','LATCHED')
        if unit(s)['recovery']['state']=='LATCHED':break
        time.sleep(.05)
    duration=time.time()-started
    assert unit(trace[-1])['recovery']['state']=='LATCHED'
    assert 2.4<=duration<=3.3,duration
    assert codes(trace[-1])==ZERO_CODES
    active=[s for s in trace if unit(s)['recovery']['state']=='ACTIVE' and .005<abs(row(s)['voltage'])<.045]
    points=sorted({s['meter']['at']:s['meter']['voltage'] for s in active}.items());assert len(points)>8
    tm=statistics.mean(t for t,v in points);vm=statistics.mean(v for t,v in points)
    slope=sum((t-tm)*(v-vm) for t,v in points)/sum((t-tm)**2 for t,v in points)
    assert .019<=abs(slope)<=.0202 and sign*slope<0,slope
    time.sleep(.4);latched=state();assert unit(latched)['recovery']['state']=='LATCHED'
    result=dict(sign=sign,duration_s=duration,slope_V_per_s=slope,meter_zero=latched['meter']['voltage'],samples=trace)
    safe_rows.append(result);save('safe_zero',safe_rows);print({k:v for k,v in result.items() if k!='samples'})
    action('network');rearmed=wait_for(lambda s:unit(s)['network']['state']=='good' and unit(s)['recovery']['state']=='READY')
    assert codes(rearmed)==ZERO_CODES
print('PASS: all four stored zero codes, TCP-independent ramp, measured slew, latch and rearm')


In [ ]:
assert max(abs(r['meter_zero']) for r in safe_rows)<=TOLERANCE_V,'Safe-zero analog endpoint exceeds ±0.5 mV'
print('PASS: absolute safe-zero endpoint accuracy')


## Final state and calibration diagnosis — no calibration is applied

In [ ]:
assert PREFLIGHT_PASSED,'DAC 10 physical preflight has not passed'
for ch in CHANNELS:action('set',channel=ch,voltage=0)
time.sleep(.4);final=state();fresh(final);save('final',compact(final))
xs=[r['displayed'] for r in voltage_rows];ys=[r['meter'] for r in voltage_rows]
xm=statistics.mean(xs);ym=statistics.mean(ys)
gain=sum((x-xm)*(y-ym) for x,y in zip(xs,ys))/sum((x-xm)**2 for x in xs)
offset=ym-gain*xm
fit=dict(gain=gain,offset_V=offset,max_residual_mV=max(abs(y-(gain*x+offset))*1000 for x,y in zip(xs,ys)),applied=False)
save('diagnosis',fit);print('Diagnostic fit only:',fit)
print('Final codes:',codes(final),'meter:',final['meter']['voltage'],'V')
assert codes(final)==ZERO_CODES

# No job in this server session may have commanded Units 1 or 2 during this acceptance run.
assert all(job['unit']==UNIT for job in final['jobs'].values()),'Unexpected command to another unit'
print('PASS: all dashboard actions targeted Unit 3; Units 1 and 2 were observed only')
